<a href="https://colab.research.google.com/github/harold-roseberg/EV-Optimisation-Algorithm-UK-/blob/main/ev_optimisation_HaroldRoseberg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# EV wholesale optimisation model

#This notebook estimates the wholesale optimisation value of an average EV using historical GB market index prices.

#Key assumptions:
#- EV consumption: 3,000 kWh/year
#- Charger: 7.4 kW
#- Battery: 60 kWh
#- Main plug-in window: 20:00–06:00, with sensitivities at 19:00 and 18:00
#- Scenarios: immediate charging, smart charging, trading, boosted trading
#- Trading requires bidirectional charging and is included as an upside case


In [ ]:
#0.1 Import the price data

!pip install elexonpy #Helpful API tool to download Elexon data
import pandas as pd
from elexonpy.api.market_index_api import MarketIndexApi
api = MarketIndexApi()

def get_mid(start, end, providers=("APXMIDP",), chunk_days=7): #Note this pulls the APX (EPEX) prices
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    chunks, t = [], start

    #The Elexon portal only handles 7 day chunk downloads, so proceeds sequentially between start and end data
    while t < end:
        u = min(t + pd.Timedelta(days=chunk_days), end)
        df = api.balancing_pricing_market_index_get(_from=t.isoformat().replace("+00:00", "Z"),
            to=u.isoformat().replace("+00:00", "Z"),data_providers=list(providers),format="dataframe",)

        if df is not None and len(df):
            chunks.append(df)
        t = u

    if not chunks:
        return pd.DataFrame()

    df = pd.concat(chunks, ignore_index=True)
    df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_").str.lower()

    df["start_time"] = pd.to_datetime(df["start_time"], utc=True)

    return (df.drop_duplicates().sort_values("start_time", ascending=True)
          .reset_index(drop=True))

# The code pulls the data between 2024 and today
df = get_mid("2024-01-01", "2026-06-29")
print(df.head(12))
print(df.columns)

In [13]:
# 0.2 Keeps only the timestamp and market price columns (others are redundant)
df_ev = df[["start_time", "price"]].copy()
df_ev["start_time"] = pd.to_datetime(df_ev["start_time"], utc=True)
df_ev = df_ev.sort_values("start_time").reset_index(drop=True)

# Half-hourly settlement period: 1-48
df_ev["hh_period"] = ((df_ev["start_time"].dt.hour * 2) + (df_ev["start_time"].dt.minute // 30) + 1)

# Hourly tag: 1-24
df_ev["hour_period"] = df_ev["start_time"].dt.hour + 1

# 0.3 The major inputs for the EV optimiser
ev_battery_size_kwh = 60    #Assumed size of average EV battery size
ev_annual_charge_kwh = 3000     #Per the take-home assumptions
charger_capacity_kw = 7.4     #Standard home charger
charging_start_hour = 20    #Per the take-home assumptions
charging_end_hour = 6     #Take-home assumptions say 8pm-midnight, but I assume it will stay connected to 6am
ev_daily_use_kwh = ev_annual_charge_kwh / 365
interval_hours = 0.5 #Accounts for half-hourly data
max_charge_per_interval_kwh = charger_capacity_kw * interval_hours
price_to_kwh_multiplier = 1 / 1000
battery_degradation_cost_gbp_per_kwh = 0.05 #My assumption on degradation cost (£100/kWh pack value, 2,000 cycles)
export_capacity_kw = charger_capacity_kw
max_export_per_interval_kwh = export_capacity_kw * interval_hours
# For the "trader" option below, sets the minimum trading threshold (anchored here on degradation cost)
trading_risk_multiplier = 0.5 #The optimised trader will trade spreads that are x% of the degradation cost
degradation_threshold_gbp_per_mwh = battery_degradation_cost_gbp_per_kwh * 1000 * trading_risk_multiplier
# For the "trader" option, sets a minimum SoC of 30% for the battery (trades that go below this aren't accepted)
customer_min_soc_pct = 0.30
customer_min_soc_kwh = ev_battery_size_kwh * customer_min_soc_pct


**The different charging algorithms**

In [16]:
import math, numpy as np

# 1.0 This creates a charging "flag" based on charging start hour and end hour
hour = df_ev["start_time"].dt.hour
df_ev["charging_window"] = ((hour >= charging_start_hour) & (hour < charging_end_hour)).astype(int) if charging_start_hour < charging_end_hour else ((hour >= charging_start_hour) | (hour < charging_end_hour)).astype(int)

def add_charge_session_date(df_ev):
    df_out = df_ev.copy(); ts = pd.to_datetime(df_out["start_time"], utc=True)
    df_out["charge_session_date"] = ts.dt.date if charging_start_hour < charging_end_hour else np.where(ts.dt.hour >= charging_start_hour, (ts + pd.Timedelta(days=1)).dt.date, ts.dt.date)
    return df_out

# 1.1 Scenario A: The EV charges immediately after being plugged in, no optimising, baseline case
def run_immediate_charging(df_ev):
    df_out = add_charge_session_date(df_ev)
    charge_counts = df_out[df_out["charging_window"] == 1].groupby("charge_session_date").size()
    bad = charge_counts[charge_counts * max_charge_per_interval_kwh < ev_daily_use_kwh]
    if len(bad): raise ValueError(f"Not enough charging hours: {list(bad.index)}. Need {ev_daily_use_kwh:.2f} kWh/day, minimum available is {(bad.min() * max_charge_per_interval_kwh):.2f} kWh.")
    non_charge_n = 48 - int(df_out["charging_window"].iloc[:48].sum()); use_per_non_charge = ev_daily_use_kwh / non_charge_n
    soc, socs, charges = ev_battery_size_kwh, [], []
    # The SOC loop below, charges at the charger capacity as soon as connected, accounts for partial charging in a period
    for _, row in df_out.iterrows():
        charge = min(max_charge_per_interval_kwh, ev_battery_size_kwh - soc) if row["charging_window"] == 1 else 0
        soc = soc + charge if row["charging_window"] == 1 else max(soc - use_per_non_charge, 0)
        charges.append(charge); socs.append(soc)
    df_out["charged_volume_kwh_immediate"] = charges; df_out["soc_kwh_immediate"] = socs; df_out["soc_pct_immediate"] = df_out["soc_kwh_immediate"] / ev_battery_size_kwh; df_out["charge_cost_immediate"] = df_out["charged_volume_kwh_immediate"] * df_out["price"] * price_to_kwh_multiplier
    return df_out

df_ev_immediate = run_immediate_charging(df_ev)

# 1.2 Scenario B: The EV charges at the cheapest times in the charging period
def run_cheapest_window_charging(df_ev):

    #The code below calculates how many charging time periods are required, then sorts them by lowest price and charges in those
    df_out = add_charge_session_date(df_ev); required_n = math.ceil(ev_daily_use_kwh / max_charge_per_interval_kwh)
    charge_counts = df_out[df_out["charging_window"] == 1].groupby("charge_session_date").size(); bad = charge_counts[charge_counts < required_n]
    if len(bad): raise ValueError(f"Not enough charging intervals: {list(bad.index)}. Need {required_n} half-hour periods/day, minimum available is {bad.min()}.")
    df_out["smart_charge_flag"] = 0; df_out["planned_smart_charge_kwh"] = 0.0

    for _, group in df_out[df_out["charging_window"] == 1].groupby("charge_session_date"):
        remaining = ev_daily_use_kwh
        for idx in group.sort_values("price").head(required_n).index:
            charge = min(max_charge_per_interval_kwh, remaining); df_out.loc[idx, "smart_charge_flag"] = 1; df_out.loc[idx, "planned_smart_charge_kwh"] = charge; remaining -= charge
            if remaining <= 0: break

    non_charge_n = 48 - int(df_out["charging_window"].iloc[:48].sum()); use_per_non_charge = ev_daily_use_kwh / non_charge_n
    soc, socs, charges = ev_battery_size_kwh, [], []
    for _, row in df_out.iterrows():
        charge = min(row["planned_smart_charge_kwh"], ev_battery_size_kwh - soc) if row["smart_charge_flag"] == 1 else 0
        soc = soc + charge if row["smart_charge_flag"] == 1 else max(soc - use_per_non_charge, 0) if row["charging_window"] == 0 else soc
        charges.append(charge); socs.append(soc)
    df_out["charged_volume_kwh_smart"] = charges; df_out["soc_kwh_smart"] = socs; df_out["soc_pct_smart"] = df_out["soc_kwh_smart"] / ev_battery_size_kwh; df_out["charge_cost_smart"] = df_out["charged_volume_kwh_smart"] * df_out["price"] * price_to_kwh_multiplier
    return df_out

df_ev_smart = run_cheapest_window_charging(df_ev)

# 1.3 Scenario C: The EV follows smart charging first, then uses remaining connected slots for trades that meet the spread threshold
def run_profit_aware_trader(df_ev):

    # First section of the code is the same as above: EV charging takes priority low-price hours
    df_out = add_charge_session_date(df_ev); required_n = math.ceil(ev_daily_use_kwh / max_charge_per_interval_kwh)
    charge_counts = df_out[df_out["charging_window"] == 1].groupby("charge_session_date").size()
    insufficient_sessions = charge_counts[charge_counts < required_n]
    if len(insufficient_sessions): raise ValueError(f"Not enough charging intervals: {list(insufficient_sessions.index)}. Need {required_n} half-hour periods/day, minimum available is {insufficient_sessions.min()}.")
    df_out["trader_action"] = "hold"; df_out["trader_charge_flag"] = 0; df_out["planned_trader_import_kwh"] = 0.0; df_out["planned_trader_export_kwh"] = 0.0

    for _, group in df_out[df_out["charging_window"] == 1].groupby("charge_session_date"):
        remaining = ev_daily_use_kwh; smart_idx = []
        for idx in group.sort_values("price").head(required_n).index:
            charge = min(max_charge_per_interval_kwh, remaining); smart_idx.append(idx)
            df_out.loc[idx, ["trader_action", "trader_charge_flag", "planned_trader_import_kwh"]] = ["smart_charge", 1, charge]; remaining -= charge
            if remaining <= 0: break

        # Remaining connected slots become import/export trade pairs, subject to the spread threshold
        # Assumes bi-directional charging, and does not model round-trip losses, imbalance, etc.
        free = group.drop(index=smart_idx); buys = list(free.sort_values("price").index); sells = list(free.sort_values("price", ascending=False).index)
        used, buy_i, sell_i = set(), 0, 0
        while buy_i < len(buys) and sell_i < len(sells):
            b, s = buys[buy_i], sells[sell_i]; spread = df_out.loc[s, "price"] - df_out.loc[b, "price"]
            if b == s or b in used: buy_i += 1; continue
            if s in used: sell_i += 1; continue
            if spread <= degradation_threshold_gbp_per_mwh: break
            df_out.loc[b, ["trader_action", "trader_charge_flag", "planned_trader_import_kwh"]] = ["trade_buy", 1, max_charge_per_interval_kwh]
            df_out.loc[s, ["trader_action", "planned_trader_export_kwh"]] = ["trade_sell", max_export_per_interval_kwh]
            used.update([b, s]); buy_i += 1; sell_i += 1

    # Same SOC loop as Scenario B, but export is limited by customer reserve plus expected driving use
    non_charge_n = 48 - int(df_out["charging_window"].iloc[:48].sum()); use_per_non_charge = ev_daily_use_kwh / non_charge_n
    trader_export_reserve_kwh = min(ev_battery_size_kwh, customer_min_soc_kwh + ev_daily_use_kwh)

    soc, socs, imports, exports = ev_battery_size_kwh, [], [], []
    for _, row in df_out.iterrows():
        imp, exp = 0.0, 0.0
        if row["charging_window"] == 0:
            soc = max(soc - use_per_non_charge, 0)
        elif row["planned_trader_import_kwh"] > 0:
            imp = min(row["planned_trader_import_kwh"], ev_battery_size_kwh - soc); soc = min(soc + imp, ev_battery_size_kwh)
        elif row["planned_trader_export_kwh"] > 0:
            exp = min(row["planned_trader_export_kwh"], max(soc - trader_export_reserve_kwh, 0)); soc = max(soc - exp, 0)
        imports.append(imp); exports.append(exp); socs.append(soc)

    df_out["charged_volume_kwh_trader"] = imports; df_out["exported_volume_kwh_trader"] = exports; df_out["net_charged_volume_kwh_trader"] = df_out["charged_volume_kwh_trader"] - df_out["exported_volume_kwh_trader"]
    df_out["soc_kwh_trader"] = socs; df_out["soc_pct_trader"] = df_out["soc_kwh_trader"] / ev_battery_size_kwh
    df_out["charge_cost_trader"] = df_out["charged_volume_kwh_trader"] * df_out["price"] * price_to_kwh_multiplier
    df_out["export_revenue_trader"] = df_out["exported_volume_kwh_trader"] * df_out["price"] * price_to_kwh_multiplier
    df_out["battery_degradation_cost_trader"] = (df_out["net_charged_volume_kwh_trader"].clip(lower=0) + df_out["exported_volume_kwh_trader"]) * battery_degradation_cost_gbp_per_kwh
    df_out["net_charge_cost_trader"] = df_out["charge_cost_trader"] - df_out["export_revenue_trader"] + df_out["battery_degradation_cost_trader"]
    if (df_out["soc_kwh_trader"] < customer_min_soc_kwh - 1e-9).any(): raise ValueError("Trader SoC went below customer minimum SoC.")
    return df_out

df_ev_trader = run_profit_aware_trader(df_ev)

**Running the code**

In [ ]:
# 2.0 Run scenarios for adjustable charging window, optional degradation, summary

# The algo runs on timing inputs, but also has a tuner for trading risk multiplier and whether to include degradation costs as a reference
def run_ev_scenario_outputs(start_hour=20, end_hour=6, include_degradation=True, trading_risk_multipliers=(1.0, 0.5), export_wide=False, download_outputs=True):
    global charging_start_hour, charging_end_hour, degradation_threshold_gbp_per_mwh, df_ev, df_ev_immediate, df_ev_smart, df_ev_trader, df_ev_scenarios_wide
    if isinstance(trading_risk_multipliers, (int, float)): trading_risk_multipliers = (trading_risk_multipliers,)

    # The below actually overrides charging times and trading hurdle for this output run
    charging_start_hour = start_hour; charging_end_hour = end_hour

    hour = df_ev["start_time"].dt.hour
    df_ev["charging_window"] = ((hour >= charging_start_hour) & (hour < charging_end_hour)).astype(int) if charging_start_hour < charging_end_hour else ((hour >= charging_start_hour) | (hour < charging_end_hour)).astype(int)
    df_ev_immediate = run_immediate_charging(df_ev); df_ev_smart = run_cheapest_window_charging(df_ev)

    # Merges the three algos into a single dataframe for ease of analysis
    base_cols = ["start_time", "price", "hh_period", "hour_period", "charging_window", "charge_session_date"]
    immediate_cols = base_cols + ["charged_volume_kwh_immediate", "soc_kwh_immediate", "soc_pct_immediate", "charge_cost_immediate"]
    smart_cols = base_cols + ["smart_charge_flag", "planned_smart_charge_kwh", "charged_volume_kwh_smart", "soc_kwh_smart", "soc_pct_smart", "charge_cost_smart"]
    trader_cols = base_cols + ["trader_charge_flag", "trader_action", "charged_volume_kwh_trader", "exported_volume_kwh_trader", "net_charged_volume_kwh_trader", "soc_kwh_trader", "soc_pct_trader", "charge_cost_trader", "export_revenue_trader", "battery_degradation_cost_trader", "net_charge_cost_trader"]

    # Sums up the various costs and power volumes for each year
    def summarise_costs(x, year_label, case, trading_risk_multiplier):
        immediate_import_cost = x["charge_cost_immediate"].sum(); smart_import_cost = x["charge_cost_smart"].sum(); trader_import_cost = x["charge_cost_trader"].sum()
        trader_export_revenue = x["export_revenue_trader"].sum()
        immediate_import_kwh = x["charged_volume_kwh_immediate"].sum(); smart_import_kwh = x["charged_volume_kwh_smart"].sum(); trader_import_kwh = x["charged_volume_kwh_trader"].sum()
        trader_export_kwh = x["exported_volume_kwh_trader"].sum(); trader_net_kwh = x["net_charged_volume_kwh_trader"].sum()
        immediate_deg = immediate_import_kwh * battery_degradation_cost_gbp_per_kwh if include_degradation else 0
        smart_deg = smart_import_kwh * battery_degradation_cost_gbp_per_kwh if include_degradation else 0
        trader_deg = (max(trader_net_kwh, 0) + trader_export_kwh) * battery_degradation_cost_gbp_per_kwh if include_degradation else 0
        immediate_total = immediate_import_cost + immediate_deg; smart_total = smart_import_cost + smart_deg; trader_total = trader_import_cost - trader_export_revenue + trader_deg
        scenario = "profit_aware_trader" if case in ["trading", "trading_boosted"] else case
        import_cost = trader_import_cost if scenario == "profit_aware_trader" else immediate_import_cost if scenario == "immediate" else smart_import_cost
        export_revenue = trader_export_revenue if scenario == "profit_aware_trader" else 0
        degradation_cost = trader_deg if scenario == "profit_aware_trader" else immediate_deg if scenario == "immediate" else smart_deg
        total_cost = trader_total if scenario == "profit_aware_trader" else immediate_total if scenario == "immediate" else smart_total
        out = pd.DataFrame({"year": [year_label], "case": [case], "scenario": [scenario], "import_cost_gbp": [import_cost], "export_revenue_gbp": [export_revenue], "degradation_cost_gbp": [degradation_cost], "total_cost_gbp": [total_cost]})
        out["total_cost_gbp_per_battery_kwh"] = out["total_cost_gbp"] / ev_battery_size_kwh
        out["saving_vs_immediate_gbp"] = immediate_total - out["total_cost_gbp"]; out["saving_vs_immediate_pct"] = out["saving_vs_immediate_gbp"] / immediate_total
        out["charging_start_hour"] = start_hour; out["charging_end_hour"] = end_hour; out["include_degradation"] = include_degradation; out["trading_risk_multiplier"] = trading_risk_multiplier; out["degradation_threshold_gbp_per_mwh"] = battery_degradation_cost_gbp_per_kwh * 1000 * trading_risk_multiplier
        return out

    comparisons, wide_outputs = [], []
    for trading_risk_multiplier in trading_risk_multipliers:
        degradation_threshold_gbp_per_mwh = battery_degradation_cost_gbp_per_kwh * 1000 * trading_risk_multiplier
        df_ev_trader = run_profit_aware_trader(df_ev)
        df_ev_scenarios_wide = df_ev_immediate[immediate_cols].merge(df_ev_smart[smart_cols], on=base_cols, how="outer").merge(df_ev_trader[trader_cols], on=base_cols, how="outer").sort_values("start_time").reset_index(drop=True)
        df_ev_scenarios_wide["year"] = df_ev_scenarios_wide["start_time"].dt.year; df_ev_scenarios_wide["trading_risk_multiplier"] = trading_risk_multiplier; df_ev_scenarios_wide["degradation_threshold_gbp_per_mwh"] = degradation_threshold_gbp_per_mwh
        case = "trading" if trading_risk_multiplier == 1.0 else "trading_boosted" if trading_risk_multiplier == 0.5 else f"trading_{trading_risk_multiplier}"

        if trading_risk_multiplier == trading_risk_multipliers[0]:
            comparisons.append(summarise_costs(df_ev_scenarios_wide, "all", "immediate", trading_risk_multiplier)); comparisons.append(summarise_costs(df_ev_scenarios_wide, "all", "smart", trading_risk_multiplier))
            for y, g in df_ev_scenarios_wide.groupby("year"):
                comparisons.append(summarise_costs(g, y, "immediate", trading_risk_multiplier)); comparisons.append(summarise_costs(g, y, "smart", trading_risk_multiplier))

        comparisons.append(summarise_costs(df_ev_scenarios_wide, "all", case, trading_risk_multiplier))
        for y, g in df_ev_scenarios_wide.groupby("year"): comparisons.append(summarise_costs(g, y, case, trading_risk_multiplier))
        if export_wide: wide_outputs.append(df_ev_scenarios_wide.copy())

    comparison = pd.concat(comparisons, ignore_index=True)

    comparison_out = f"ev_charging_comparison_{start_hour}_{end_hour}_deg_{include_degradation}_multi_risk.csv"
    comparison.to_csv(comparison_out, index=False)
    if download_outputs: files.download(comparison_out)
    if export_wide:
        df_ev_scenarios_wide = pd.concat(wide_outputs, ignore_index=True)
        wide_out = f"ev_scenarios_wide_{start_hour}_{end_hour}_deg_{include_degradation}_multi_risk.csv"; df_ev_scenarios_wide.to_csv(wide_out, index=False)
        if download_outputs: files.download(wide_out)

    print(comparison)
    return comparison, df_ev_scenarios_wide

#Making the requests across three scenarios (plug-in at 8pm, 7pm and 6pm)
comparison_20, df_ev_scenarios_wide_20 = run_ev_scenario_outputs(start_hour=20, end_hour=6, include_degradation=True, trading_risk_multipliers=(1.0, 0.5), export_wide=False, download_outputs=False)
comparison_19, df_ev_scenarios_wide_19 = run_ev_scenario_outputs(start_hour=19, end_hour=6, include_degradation=True, trading_risk_multipliers=(1.0, 0.5), export_wide=False, download_outputs=False)
comparison_18, df_ev_scenarios_wide_18 = run_ev_scenario_outputs(start_hour=18, end_hour=6, include_degradation=True, trading_risk_multipliers=(1.0, 0.5), export_wide=False, download_outputs=False)